# Module 12: Placebo Tests

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

A placebo test runs the real analysis somewhere the answer is known to be
zero. If it returns an effect, the analysis produces effects out of nothing
and the real estimate cannot be trusted.

Three kinds, all cheap, all skipped.

**About 25 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

keep = [a for a in TRAINED if a != "A007"]
d = f[f["agency_id"].isin(keep + COMPARISON)].copy()
d["lo"] = np.log(d["n_arrests"])
d["tr"] = d["agency_id"].isin(keep).astype(float)
d["settled"] = ((d["tr"] == 1) & (d["period"] == "after")).astype(float)
d["phase"] = ((d["tr"] == 1) & (d["period"] == "phase")).astype(float)

pct = lambda b: 100 * (np.exp(b) - 1)


def effect(outcome, extra="", offset=None):
    form = f"{outcome} ~ C(agency_id) + C(year_month) + settled + phase{extra}"
    z = smf.glm(form, d, family=sm.families.Poisson(), offset=offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

## 2. Fake dates

Pretend the program started on a date inside the pre period, when nothing
happened, and run the same estimator.

In [ ]:
pre = d[d["period"] == "before"].copy()
rows = []
for cut in ["2020-07", "2021-01", "2021-07", "2022-01", "2022-07"]:
    s = pre.copy()
    s["fake"] = ((s["tr"] == 1) & (s["year_month"] >= cut)).astype(float)
    z = smf.glm("n_uof ~ C(agency_id) + C(year_month) + fake", s,
                family=sm.families.Poisson(), offset=s["lo"]).fit()
    lo, hi = z.conf_int().loc["fake"]
    rows.append({"fake start date": cut,
                 "estimate": f"{pct(z.params['fake']):+.1f}%",
                 "95 percent interval": f"[{pct(lo):+.1f}, {pct(hi):+.1f}]",
                 "covers zero": "yes" if lo < 0 < hi else "NO"})
print("  the program did not exist on any of these dates. The truth is 0.0%\n")
pd.DataFrame(rows).set_index("fake start date")

Every interval covers zero, which is the pass.

**The point estimates are not zero, and they drift.** They run from 0.8
percent at the earliest date to 4.6 at the latest, and the drift is the
residual difference in pre trends between the two groups, 0.71 percent a
year from [Module 7](Module_07_Testing_Parallel_Trends.ipynb), accumulating
over a longer fake post period.

A placebo that comes back at exactly zero would be surprising. What matters
is whether the drift is small next to the real estimate, and 4.6 against 12.6
is on the edge of comfortable. **Report it rather than declaring a pass.**

## 3. Fake outcomes

Run the real analysis, on the real dates, against an outcome the program
should not touch.

In [ ]:
rows = []
for label, outcome, off in [("use of force per arrest, the real outcome",
                             "n_uof", d["lo"]),
                            ("arrests", "n_arrests", None),
                            ("calls for service", "total_cfs", None)]:
    e, lo, hi, _ = effect(outcome, offset=off)
    rows.append({"outcome": label, "estimate": f"{e:+.2f}%",
                 "95 percent interval": f"[{lo:+.2f}, {hi:+.2f}]",
                 "covers zero": "yes" if lo < 0 < hi else "NO"})
pd.DataFrame(rows).set_index("outcome")

Arrests pass cleanly. **Calls for service fail**, by 0.42 percent with an
interval from 0.12 to 0.73.

That is the right moment to look at the size rather than the label. A 0.42
percent change in call volume is not a plausible route to a 12.6 percent
change in use of force, and with tens of thousands of calls a month the test
can detect differences far below anything that matters.

**A placebo test with enormous power fails on noise.** Report the estimate
and the interval, and say whether the size is consistent with a real
pathway. Do not report "the placebo failed" and stop.

## 4. Fake treatment groups

The strongest of the three. Reassign the treatment label at random many
times, and see where the real estimate sits in the distribution of estimates
the procedure produces from nothing.

In [ ]:
rng = np.random.default_rng(21)
pool = sorted(d["agency_id"].unique())
real = effect("n_uof", offset=d["lo"])[0]

fakes = []
for _ in range(400):
    pick = list(rng.choice(pool, len(keep), replace=False))
    s = d.copy()
    s["settled"] = ((s["agency_id"].isin(pick)) & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(pick)) & (s["period"] == "phase")).astype(float)
    z = smf.glm("n_uof ~ C(agency_id) + C(year_month) + settled + phase", s,
                family=sm.families.Poisson(), offset=s["lo"]).fit()
    fakes.append(pct(z.params["settled"]))
fakes = np.array(fakes)

print(f"  400 random reassignments of the treatment label:")
print(f"    median {np.median(fakes):+.1f}%, middle 90 percent from "
      f"{np.percentile(fakes, 5):+.1f}% to {np.percentile(fakes, 95):+.1f}%")
print(f"\n  the real estimate: {real:+.1f}%")
print(f"  random draws at least this negative: "
      f"{100 * np.mean(fakes <= real):.1f} percent")

This is a **randomisation test**, and it makes no assumption about the shape
of the sampling distribution. It asks a simple question: among all the ways
five agencies could have been labelled treated, how unusual is the one that
actually was?

It is the right test when there are few units, because the model based
interval relies on asymptotics that eleven agencies do not supply. Time
Series Advanced [Module 10](../../../Time_Series/Advanced/Module_10_Panel_And_Hierarchical.md)
makes the same point about clustering.

## 5. Which placebo to run

| Test | Catches | Cost |
|---|---|---|
| Fake dates | an estimator that finds effects in any window | minutes |
| Fake outcomes | an effect that is really something else changing | minutes |
| Fake treatment groups | an inference that is too confident for the sample size | a few minutes of compute |

Run all three. Report all three, including the one that half failed.

## Exercise

The fake date test used only the pre period. Run it on a date **after** the
program, which is a different and sharper question.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    post = d[d["period"] == "after"].copy()
    rows = []
    for cut in ["2024-06", "2024-12", "2025-06"]:
        s = post.copy()
        s["fake"] = ((s["tr"] == 1) & (s["year_month"] >= cut)).astype(float)
        z = smf.glm("n_uof ~ C(agency_id) + C(year_month) + fake", s,
                    family=sm.families.Poisson(), offset=s["lo"]).fit()
        lo, hi = z.conf_int().loc["fake"]
        rows.append({"a second fake start, inside the treated period": cut,
                     "estimate": f"{pct(z.params['fake']):+.1f}%",
                     "95 percent interval": f"[{pct(lo):+.1f}, {pct(hi):+.1f}]"})
    print("  the program was already fully in place throughout. "
          "A second start should show nothing.\n")
    display(pd.DataFrame(rows).set_index(
        "a second fake start, inside the treated period"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

All three intervals cover zero, which is the pass. The program was fully in
place across the whole window, so a further step change at an arbitrary later
date should find nothing, and it does.

**This version tests something the pre period placebo cannot.** A pre period
placebo asks whether the two groups were diverging before the program. This
one asks whether the effect, once established, is stable, or whether the
estimate is being driven by one stretch of months. An effect that appears
only in the last six months of a 30 month window would show up here and
nowhere else.

It is the placebo closest in spirit to the event study, and it is the one
worth running when a reader asks whether the effect persisted.

</details>

---

**Next:** [Module 13: Spillover and Contamination](Module_13_Spillover_And_Contamination.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*